# Crosscoders: Analyzing Cross-Layer Features

**Survey Reference:** Section 6.5.3 - Crosscoders: Analyzing Cross-Layer Features

## Overview

A **crosscoder** learns sparse features by reading activations from **multiple layers simultaneously**:

$$\hat{x}_{\ell,t} = W_{\text{out}} z_{\ell,t} + b_{\text{out}}$$

$$z_{\ell,t} = \text{ReLU}(W_{\text{in}} [h_{\ell',t}]_{\ell' \in \mathcal{L}} + b_{\text{in}})$$

where $z_{\ell,t} \in \mathbb{R}^{d_{\text{sparse}}}$ encodes sparse features constructed from activations across a set of layers $\mathcal{L}$ at token position $t$.


Crosscoders **resolve cross-layer superposition** by tracking features that persist through the residual stream. Instead of learning duplicate representations of the same concept at each layer, crosscoders extract features that span multiple layers, simplifying circuit analysis.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

### Crosscoder

A crosscoder has:
- **Shared encoder $W_{\text{in}}$:** Maps concatenated layer activations to sparse features
- **Shared decoder $W_{\text{out}}$:** Reconstructs activations for each layer from the same features
- **Layer-specific biases:** Allow per-layer reconstruction adjustments

### Formulation

Given activations from layers $\mathcal{L} = \{\ell_1, \ell_2, ..., \ell_k\}$ at position $t$:

**Encoding:**
$$z_{\ell,t} = \text{ReLU}(W_{\text{in}} [h_{\ell_1,t}; h_{\ell_2,t}; ...; h_{\ell_k,t}] + b_{\text{in}})$$

**Decoding:**
$$\hat{x}_{\ell,t} = W_{\text{out}} z_{\ell,t} + b_{\text{out}}^{(\ell)}$$

where:
- $[h_{\ell',t}]_{\ell' \in \mathcal{L}}$ is the concatenation of activations across layers
- $z_{\ell,t}$ is the sparse feature representation
- $b_{\text{out}}^{(\ell)}$ provides layer-specific adjustments

**Key Difference from Per-Layer SAEs:** The encoder reads from multiple layers simultaneously, creating a unified feature space.

In [ ]:
class Crosscoder(nn.Module):
    """
    Crosscoder: Extract shared sparse features across multiple layers.
    
    Resolves cross-layer superposition by tracking features that persist
    through the residual stream.
    
    Architecture:
        z = ReLU(W_in @ [h_l1, h_l2, ..., h_lk] + b_in)
        x_hat_l = W_out @ z + b_out_l
    """
    
    def __init__(
        self, 
        d_model: int, 
        layers: list[int],  # Specific layers to read from
        d_sparse: int,
        causal: bool = True
    ):
        """
        Args:
            d_model: Model hidden dimension
            layers: List of layer indices to read from (e.g., [4, 8, 12, 16])
            d_sparse: Number of sparse features to learn
            causal: If True, only read from layers <= current layer
        """
        super().__init__()
        self.d_model = d_model
        self.layers = sorted(layers)
        self.d_sparse = d_sparse
        self.causal = causal
        
        # Shared encoder: concatenates activations from all layers
        input_dim = d_model * len(layers)
        self.W_in = nn.Parameter(torch.randn(input_dim, d_sparse) / np.sqrt(input_dim))
        self.b_in = nn.Parameter(torch.zeros(d_sparse))
        
        # Shared decoder with per-layer output biases
        self.W_out = nn.Parameter(torch.randn(d_sparse, d_model) / np.sqrt(d_sparse))
        self.b_out = nn.ParameterDict({
            str(layer): nn.Parameter(torch.zeros(d_model))
            for layer in layers
        })
        
    def encode(self, layer_activations: dict[int, torch.Tensor]) -> torch.Tensor:
        """
        Encode multi-layer activations to shared sparse features.
        
        Args:
            layer_activations: Dict mapping layer_idx -> activations [batch, seq, d_model]
            
        Returns:
            z: Sparse feature activations [batch, seq, d_sparse]
        """
        # Concatenate activations from specified layers
        acts_list = [layer_activations[layer] for layer in self.layers]
        h_concat = torch.cat(acts_list, dim=-1)  # [batch, seq, d_model * n_layers]
        
        # Encode to sparse features
        z = F.relu(h_concat @ self.W_in + self.b_in)  # [batch, seq, d_sparse]
        return z
    
    def decode(self, z: torch.Tensor, target_layer: int) -> torch.Tensor:
        """
        Decode sparse features to reconstruct activations for a specific layer.
        
        Args:
            z: Sparse features [batch, seq, d_sparse]
            target_layer: Layer to reconstruct
            
        Returns:
            x_hat: Reconstructed activations [batch, seq, d_model]
        """
        x_hat = z @ self.W_out + self.b_out[str(target_layer)]
        return x_hat
    
    def forward(
        self, 
        layer_activations: dict[int, torch.Tensor]
    ) -> tuple[dict[int, torch.Tensor], torch.Tensor]:
        """
        Full forward pass: encode and decode all layers.
        
        Args:
            layer_activations: Dict mapping layer_idx -> activations
            
        Returns:
            reconstructions: Dict mapping layer_idx -> reconstructed activations
            z: Sparse feature activations
        """
        # Encode from all layers
        z = self.encode(layer_activations)
        
        # Decode to each layer
        reconstructions = {
            layer: self.decode(z, layer)
            for layer in self.layers
        }
        
        return reconstructions, z
    
    def get_feature_norms_per_layer(self) -> dict[int, torch.Tensor]:
        """
        Analyze how much each feature contributes to each layer.
        
        Returns:
            Dict mapping layer_idx -> feature contribution norms [d_sparse]
        """
        norms = {}
        for layer in self.layers:
            # For each feature, get its decoder column and compute norm
            decoder_cols = self.W_out.T  # [d_sparse, d_model]
            feature_norms = torch.norm(decoder_cols, dim=1)  # [d_sparse]
            norms[layer] = feature_norms
        return norms